Pytorch的nn.MSELoss()代替平方损失函数(损失函数就是体现真实值与预测值的差距，越小越好)
*         data.DataLoader 代替数据加载器
*         optim.SGD代替优化器(用于反向传播，类似于优化y=wx中的权重)
*         nn.Linear代替假设函数(类似于预测房贷函数y=wx+b)

In [2]:
#导入相关模块
import torch
from docutils.nodes import label
from holoviews.examples.gallery.apps.bokeh.crossfilter import color
from torch.utils.data import TensorDataset      #构造数据集对象
from torch.utils.data import DataLoader         #数据加载器
from torch import nn                            #nn模块中有假设函数和平方损失函数
from torch import optim                         #optim模块中有优化器函数
from sklearn.datasets import make_regression    #创建线性回归模型数据集
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']    #正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False      #用来正常显示负号

In [8]:
#1.创建数据集
x,y,coef=make_regression(
    n_samples=100,  #创造的数据有多少样本点
    n_features=1,   #特征数量
    noise=10,       #如果不加噪声，画出的拟合回归线完美穿过所有的点，不接近实际情况
    coef=True,      #代表coef是权重，返回创造这些点用的真实权重，用于比较预测权重和真实权重
    bias=14.5,      #偏执，随便写
    random_state=3
)
x=torch.tensor(x,dtype=torch.float32)
y=torch.tensor(y,dtype=torch.float32)
x
y

tensor([ 22.5139, -11.2928, -65.3266,  11.4912,  34.3622,  -4.7371,  55.7981,
         34.1948, -21.1198,   8.8681,   4.8401, -12.6738,  27.7491, -10.2920,
        -20.5800, -34.3397,  37.0163,  57.3231,   6.9397,  29.9672,  53.9756,
          4.4474,  20.7192,  -8.5406,   0.7447, -22.5050,   4.0683, -13.3933,
        -24.1197,  46.6569,  41.9460,  35.2172,   9.7509,  20.5186,  -3.9652,
          1.3670,   4.0815, -25.0052,  52.6946,  -5.1169, -14.0601,  26.0171,
        -16.5974,  65.3023, -37.8152,  37.5357,  31.6372,   0.7058,  -2.9643,
         17.8506,  15.1214,   6.6686,   4.1857,  16.1612, -72.6875,  44.3273,
         54.4379, -14.8631,  46.8485,  27.7344,  48.8398, -20.5679, -49.2582,
        -10.4936,  22.0268,   7.2506,   0.7285,  67.7584,  -7.8556,  26.0243,
         -3.9790,  58.8114,  36.1283, -12.2570,  -9.5121, -25.5052,  10.3903,
        -24.5829, -24.2417, -22.2563,  72.4724,  36.7423,   7.3779,  38.3146,
        -25.6796,  46.2081,  23.3987,  69.0189,  -0.8138,  31.38

* numpy对象 -> 张量Tensor -> 数据集对象TensorDataset -> 数据加载器DataLoader

In [7]:
#2.模型训练
# 1).创建数据集对象啊，把tensor ->数据集对象 -> 数据加载器
dataset=TensorDataset(x,y)
# 2).创建数据集对象啊，把tensor ->数据集对象 -> 数据加载器
    #数据加载器有一个特点就是分批训练，一批16个，总共7批，最后一批是4个
dataloader=DataLoader(dataset,batch_size=16,shuffle=True)
            #参1.数据集对象 参2.伦次大小   参3.是否打乱数据（因为有很多轮，如果每轮每批都一样就没意义）
# 3).创建初始的 线性回归模型
model=nn.Linear(1,1)
            #参1.输出特征有几个 参2.标签有几个
# 4).创建损失函数对象
criterion = nn.MSELoss()
# 5).创建优化器对象
optimizer=optim.SGD(model.parameters(),lr=0.01)
        #参1.模型参数(告诉优化器model中的偏置和权重需要改)   参2.学习率
# 6).具体的训练过程
    #定义变量，分别表示：训练轮数，每轮的平均损失值，训练总损失值，训练样本数
epochs,loss_list,total_loss,total_sample=100,[],0.0,0
#开始训练，按轮训练
for epoch in range(epochs):
    #每轮是按照批次训练的，所以需要从数据加载器中获取批次数据
    for train_x,train_y in dataloader: #7批 6个16 1个4
        #模型预测
        y_pred=model(train_x)
        #计算损失
        loss=criterion(y_pred,train_y.reshape(-1,1))
                #-1自动计算，所以的数变成一列，因为x是多行一列，
        #计算损失和样本批次数
        total_loss=total_loss+loss.item() #item用于将只包含一个数字的Tensor，变成数字
        total_sample=total_sample+1
        #梯度清零+反向传播+梯度更新
        optimizer.zero_grad()
        #梯度清零，pytorch,梯度会累加，上次计算为-0.3，这次0.1，第二次梯度为-0.2，所以要清零
        loss.backward()
        #反向传播，由wx求预测值->损失(相减平方)->w为自变量 -> 对损失函数求导 -> 带入x求新的梯度->新w=旧w-学习率*梯度
        optimizer.step() #梯度更新
    #把本轮的平均损失，添加到列表中
    loss_list.append(total_loss/total_sample)
    print(f'轮数：{epoch+1},平均损失值：{total_loss/total_sample}')
# 7).打印最终训练结果
print(f'{epochs}轮的平均损失分别为：{loss_list}')
print(f'模型参数，权重：{model.weight},偏置：{model.bias}')
# 8).绘制损失曲线
plt.plot(range(epochs),loss_list)
plt.title('损失值曲线变化图')
plt.grid()
plt.show()
# 9).绘制预测值与真实值的关系
# 9.1)绘制样本点的分布情况
plt.scatter(x,y)
# 9.2)绘制训练模型的预测值
y_pred=torch.tensor(data=[v*model.weight+model.bias for v in x]) #语法糖 等同for循环只有一个语句
# 9.3).计算真实值
y_true=torch.tensor(data=[v*coef+14.5 for v in x])
# 9.4).绘制真实值与预测值的折线图
plt.plot(x,y_pred,color='red',label='预测值')
plt.plot(x,y_true,color='blue',label='真实值')
# 9.5)图例，网格
plt.legend()
plt.grid()
#9.6)显示图像
plt.show()

轮数：1,平均损失值：972.9874703543527
轮数：2,平均损失值：884.8878740583148
轮数：3,平均损失值：758.4805007207962
轮数：4,平均损失值：674.7524806431362
轮数：5,平均损失值：610.0919350760323
轮数：6,平均损失值：553.0481196812221
轮数：7,平均损失值：505.72292748276067
轮数：8,平均损失值：468.3918037414551
轮数：9,平均损失值：434.331045725989
轮数：10,平均损失值：405.3064477103097
轮数：11,平均损失值：379.4623826757654
轮数：12,平均损失值：359.84330086481003
轮数：13,平均损失值：340.3046319144113
轮数：14,平均损失值：324.1311445820088
轮数：15,平均损失值：308.5746208190918
轮数：16,平均损失值：294.83309752600536
轮数：17,平均损失值：282.52934903056683
轮数：18,平均损失值：271.31222149682424
轮数：19,平均损失值：261.1424792512019
轮数：20,平均损失值：252.17670023100717
轮数：21,平均损失值：244.1085296358381
轮数：22,平均损失值：236.84799201147896
轮数：23,平均损失值：230.10919987340893
轮数：24,平均损失值：224.16704856214068
轮数：25,平均损失值：218.2349095753261
轮数：26,平均损失值：213.00190759491136
轮数：27,平均损失值：208.60847494841883
轮数：28,平均损失值：203.9426316777054
轮数：29,平均损失值：199.60384112860768
轮数：30,平均损失值：195.74325526555378
轮数：31,平均损失值：192.07547847466535
轮数：32,平均损失值：188.84539906893457
轮数：33,平均损失值：185.50297103402934
轮数：3